# Notebook de Referência - Atividade Extra (T5)

Usar as seções como guia.

**Aluno:** Rodolfo Vinicius Cima Takemoto  
**Disciplina:** Processamento de Linguagem Natural (PLN)  
**Professor:** Leonardo de Lellis Rossi  
**Atividade:** Treinamento do Modelo T5 (PTT5) para Tradução Inglês-Português (OPUS-100)

## Instruções

Neste colab iremos treinar um modelo T5 para traduzir de inglês para português. Iremos treiná-lo com o dataset OPUS-100.

- Usaremos o dataset OPUS-100 Inglês-Português. Truncamos o dataset de treino para apenas 100k pares para deixar o treinamento mais rápido. Quem quiser pode treinar com mais amostras. Se demorar muito para treinar, truncar o dataset ainda mais.

- Usaremos o BLEU como métrica. Usaremos o SacreBLEU pois sempre faz o mesmo pré-processamento (tokenização, lowercase). Não usaremos torchnlp.metrics.bleu, torchtext.data.metrics.bleu_score, etc. SacreBLEU é lento: usar poucas amostras de validação (ex: 5k)


Usaremos o modelo PTT5 disponível no model hub da HuggingFace:

https://huggingface.co/unicamp-dl/ptt5-small-portuguese-vocab

Este é  um T5 pré-treinado em textos em português e com tokenizador em português.


In [ ]:
# Configurações gerais
model_name = "unicamp-dl/ptt5-small-portuguese-vocab"
batch_size = 64
accumulate_grad_batches = 2
source_max_length = 128
target_max_length = 128
learning_rate = 1e-3

In [ ]:
! pip install sacrebleu
! pip install pytorch-lightning==1.9.5
! pip install transformers
! pip install sentencepiece

In [ ]:
!pip install nvidia-ml-py


In [ ]:
# Importar todos os pacotes de uma só vez para evitar duplicados ao longo do notebook.
import gzip
import pynvml
import os
import pytorch_lightning as pl
import random
import sacrebleu
import torch
import torch.nn.functional as F

from google.colab import drive

from pytorch_lightning.callbacks import ModelCheckpoint

from transformers import T5ForConditionalGeneration
from transformers import T5Tokenizer
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from typing import Dict
from typing import List
from typing import Tuple

/usr/local/lib/python3.13/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [ ]:
# Important: Fix seeds so we can replicate results
seed = 123
random.seed(seed)
# np.random.seed(seed)
torch.random.manual_seed(seed)
torch.cuda.manual_seed(seed)

DICA para modelos reais: Um modelo otimizado deve manter o uso de GPU próximo a 100% durante o treino.
Vamos utilizar a bilioteca abaixo para monitorar isso. Note que no modelo simples utilizado aqui o uso não vai chegar a 100%.

In [ ]:
print(f"Pytorch Lightning Version: {pl.__version__}")
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
info = pynvml.nvmlDeviceGetMemoryInfo(handle)
print(f"Memória Total: {info.total / 1024**2:.0f} MB")

Pytorch Lightning Version: 1.9.5
Memória Total: 15360 MB


Iremos salvar os checkpoints (pesos do modelo) no google drive, para que possamos continuar o treino de onde paramos.

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Preparando Dados

Primeiro, fazemos import do dataset:

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset
import random


## Carregando o dataset

Criaremos uma divisão de treino (100k pares) e val (5k pares) artificialmente.

Nota: Evitar de olhar ao máximo o dataset de teste para não ficar enviseado no que será testado. Em aplicações reais, o dataset de teste só estará disponível no futuro, ou seja, é quando o usuário começa a testar o seu produto.

In [ ]:
print("Carregando dataset OPUS-100...")
# No opus100 oficial do Helsinki-NLP, o nome da configuração é o par de línguas separado por hífen
dataset_iter = load_dataset("Helsinki-NLP/opus-100", "en-pt", split='train', streaming=True)

# 2. Converte para o formato de lista [source, target]
text_pairs = []
for i, item in enumerate(dataset_iter):
    # O formato do opus100 é {'translation': {'en': '...', 'pt': '...'}}
    en_text = item['translation']['en']
    pt_text = item['translation']['pt']
    text_pairs.append([en_text, pt_text])

    if i >= 110000:
        break

# 3. Embaralha e divide
random.seed(42)
text_pairs_list = list(text_pairs) # Garante que é uma lista para o shuffle
random.shuffle(text_pairs_list)

x_test = text_pairs_list[:5000]
x_train_full = text_pairs_list[5000:]

x_train = x_train_full[:100000]
x_val = x_train_full[100000:105000]

# --- Exibição dos resultados ---
for set_name, x in [('treino', x_train), ('validação', x_val), ('test', x_test)]:
    print(f'\n{len(x)} amostras de {set_name}')
    if len(x) > 0:
        print(f'3 primeiras amostras {set_name}:')
        for i, (source, target) in enumerate(x[:3]):
            print(f'{i}: source: {source}\n   target: {target}')

Carregando dataset OPUS-100...



100000 amostras de treino
3 primeiras amostras treino:
0: source: ==References==* Hanssens, M. 2005.
   target: ==Referências==* Hanssens, M. 2005.
1: source: Not a day goes by I don't think about you.
   target: Não passa um único dia sem que eu pense em ti.
2: source: Like, but he was explaining it to us like joyful.
   target: Estava a explicar-nos aquilo de uma forma alegre.

5000 amostras de validação
3 primeiras amostras validação:
0: source: Soares married Maria de Jesus Barroso Soares, an actress, on 22 February 1949, while in the Aljube prison, at the 3rd Conservatory of the Civil Register of Lisbon.
   target: Foi, aliás, na prisão, embora com registo na 3.ª Conservatória do Registo Civil de Lisboa, que casou, a 22 de Fevereiro de 1949, com Maria Barroso.
1: source: ↓ 0.28
   target: ↓0, 28
2: source: I'm gonna have to take it out kamikaze style.
   target: Vou ter de avançar à kamikaze.

5000 amostras de test
3 primeiras amostras test:
0: source: We have not forgotten that 

## Carregando Modelo

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(model_name)

## Criando Dataset


In [ ]:
class MyDataset(Dataset):
    def __init__(self, text_pairs: List[Tuple[str]], tokenizer,
                 source_max_length: int = 32, target_max_length: int = 32):
        self.tokenizer = tokenizer
        self.text_pairs = text_pairs
        self.source_max_length = source_max_length
        self.target_max_length = target_max_length

    def __len__(self):
        return len(self.text_pairs)

    def __getitem__(self, idx):
        source, target = self.text_pairs[idx]

        source_encoding = self.tokenizer(
            source,
            max_length=self.source_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        target_encoding = self.tokenizer(
            target,
            max_length=self.target_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        source_token_ids = source_encoding['input_ids'].squeeze(0)
        source_mask = source_encoding['attention_mask'].squeeze(0)
        target_token_ids = target_encoding['input_ids'].squeeze(0)
        target_mask = target_encoding['attention_mask'].squeeze(0)

        return source_token_ids, source_mask, target_token_ids, target_mask, source, target

## Testando o DataLoader

In [ ]:
text_pairs = [('we like pizza', 'eu gosto de pizza')]
dataset_debug = MyDataset(
    text_pairs=text_pairs,
    tokenizer=tokenizer,
    source_max_length=source_max_length,
    target_max_length=target_max_length)

dataloader_debug = DataLoader(dataset_debug, batch_size=10, shuffle=True,
                              num_workers=0)

source_token_ids, source_mask, target_token_ids, target_mask, _, _ = next(iter(dataloader_debug))
print('source_token_ids:\n', source_token_ids)
print('source_mask:\n', source_mask)
print('target_token_ids:\n', target_token_ids)
print('target_mask:\n', target_mask)

print('source_token_ids.shape:', source_token_ids.shape)
print('source_mask.shape:', source_mask.shape)
print('target_token_ids.shape:', target_token_ids.shape)
print('target_mask.shape:', target_mask.shape)

source_token_ids:
 tensor([[  31, 1528, 1079,  634, 1241, 7531,    1,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])
source_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0,

## Criando DataLoaders de Treino/Val/Test

In [ ]:
dataset_train = MyDataset(text_pairs=x_train,
                          tokenizer=tokenizer,
                          source_max_length=source_max_length,
                          target_max_length=target_max_length)

dataset_val = MyDataset(text_pairs=x_val,
                        tokenizer=tokenizer,
                        source_max_length=source_max_length,
                        target_max_length=target_max_length)

dataset_test = MyDataset(text_pairs=x_test,
                         tokenizer=tokenizer,
                         source_max_length=source_max_length,
                         target_max_length=target_max_length)

train_dataloader = DataLoader(dataset_train, batch_size=batch_size,
                              shuffle=True, num_workers=0)

val_dataloader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False,
                            num_workers=0)

test_dataloader = DataLoader(dataset_test, batch_size=batch_size,
                             shuffle=False, num_workers=0)

## Criando o T5 com Pytorch Lightning

In [ ]:
class T5Finetuner(pl.LightningModule):

    def __init__(self, tokenizer, train_dataloader, val_dataloader,
                 test_dataloader, learning_rate, target_max_length=32):
        super(T5Finetuner, self).__init__()

        self._train_dataloader = train_dataloader
        self._val_dataloader = val_dataloader
        self._test_dataloader = test_dataloader

        self.model = T5ForConditionalGeneration.from_pretrained(model_name)

        self.tokenizer = tokenizer
        self.learning_rate = learning_rate
        self.target_max_length = target_max_length

    def forward(self, source_token_ids, source_mask, target_token_ids=None,
                target_mask=None):
        if target_token_ids is not None:
            # Substitui os tokens de pad por -100 para desconsiderá-los no cálculo da loss
            labels = target_token_ids.clone()
            labels[labels == self.tokenizer.pad_token_id] = -100
            return self.model(
                input_ids=source_token_ids,
                attention_mask=source_mask,
                labels=labels
            )
        else:
            return self.model.generate(
                input_ids=source_token_ids,
                attention_mask=source_mask,
                max_length=self.target_max_length
            )

    def training_step(self, batch, batch_nb):
        source_token_ids, source_mask, target_token_ids, target_mask, _, _ = batch
        outputs = self(source_token_ids, source_mask, target_token_ids, target_mask)
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_nb):
        source_token_ids, source_mask, target_token_ids, target_mask, source_texts, target_texts = batch
        outputs = self(source_token_ids, source_mask, target_token_ids, target_mask)
        loss = outputs.loss

        # Geração de predições para cálculo do SacreBLEU
        pred_token_ids = self.model.generate(
            input_ids=source_token_ids,
            attention_mask=source_mask,
            max_length=self.target_max_length
        )
        preds = self.tokenizer.batch_decode(pred_token_ids, skip_special_tokens=True)

        return {
            'val_loss': loss,
            'preds': preds,
            'targets': list(target_texts)
        }

    def test_step(self, batch, batch_nb):
        source_token_ids, source_mask, target_token_ids, target_mask, source_texts, target_texts = batch
        outputs = self(source_token_ids, source_mask, target_token_ids, target_mask)
        loss = outputs.loss

        # Geração de predições para cálculo do SacreBLEU
        pred_token_ids = self.model.generate(
            input_ids=source_token_ids,
            attention_mask=source_mask,
            max_length=self.target_max_length
        )
        preds = self.tokenizer.batch_decode(pred_token_ids, skip_special_tokens=True)

        return {
            'test_loss': loss,
            'preds': preds,
            'targets': list(target_texts)
        }

    def validation_epoch_end(self, outputs):
        avg_loss = torch.stack([x['val_loss'] for x in outputs]).mean()

        all_preds = []
        all_targets = []
        for x in outputs:
            all_preds.extend(x['preds'])
            all_targets.extend(x['targets'])

        bleu = sacrebleu.corpus_bleu(all_preds, [all_targets])
        bleu_score = bleu.score

        self.log('avg_val_loss', avg_loss, prog_bar=True)
        self.log('avg_val_bleu', bleu_score, prog_bar=True)

        print(f"\n[Validation Epoch {self.current_epoch}] Loss: {avg_loss:.4f} | SacreBLEU: {bleu_score:.2f}")
        if len(all_preds) > 0:
            print(f"  Exemplo Alvo:     {all_targets[0]}")
            print(f"  Exemplo Predição: {all_preds[0]}")

    def test_epoch_end(self, outputs):
        avg_loss = torch.stack([x['test_loss'] for x in outputs]).mean()

        all_preds = []
        all_targets = []
        for x in outputs:
            all_preds.extend(x['preds'])
            all_targets.extend(x['targets'])

        bleu = sacrebleu.corpus_bleu(all_preds, [all_targets])
        bleu_score = bleu.score

        self.log('avg_test_loss', avg_loss, prog_bar=True)
        self.log('avg_test_bleu', bleu_score, prog_bar=True)

        print(f"\n{'='*30} RESULTADOS DE TESTE {'='*30}")
        print(f"Test Loss: {avg_loss:.4f} | Test SacreBLEU: {bleu_score:.2f}")
        print(f"{'='*75}\n")
        for i in range(min(5, len(all_preds))):
            print(f"Amostra {i+1}:")
            print(f"  Alvo (Referência): {all_targets[i]}")
            print(f"  Predição Modelo:   {all_preds[i]}\n")

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            [p for p in self.parameters() if p.requires_grad],
            lr=self.learning_rate, eps=1e-08)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=1.0)  # This is the same as no LR decay.
        return {'optimizer': optimizer, 'lr_scheduler': scheduler, 'monitor': 'avg_val_bleu'}

    def train_dataloader(self):
        return self._train_dataloader

    def val_dataloader(self):
        return self._val_dataloader

    def test_dataloader(self):
        return self._test_dataloader

In [ ]:
model = T5Finetuner(tokenizer=tokenizer,
                    train_dataloader=train_dataloader,
                    val_dataloader=val_dataloader,
                    test_dataloader=test_dataloader,
                    learning_rate=learning_rate,
                    target_max_length=target_max_length)

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

## Testando rapidamente o modelo em treino, validação e teste com um batch

In [ ]:
def gpu_usage():
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)

    used = info.used / 1024**2
    total = info.total / 1024**2

    return f"{used:.0f}/{total:.0f} MB"

In [ ]:
trainer = pl.Trainer(gpus=1,
                     precision=16,
                     enable_checkpointing=False,  # Disable checkpoint saving.
                     fast_dev_run=True)
trainer.fit(model)
trainer.test(model)
del model  # Para não ter estouro de mémoria da GPU

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=1)` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=1)` instead.
  rank_zero_deprecation(
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.uti

Training: 0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/module.py:1671: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()  # type: ignore[call-arg]


Validation: 0it [00:00, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1` reached.



--- Exemplos de validação ---

Source: Soares married Maria de Jesus Barroso Soares, an actress, on 22 February 1949, while in the Aljube prison, at the 3rd Conservatory of the Civil Register of Lisbon.
Target: Foi, aliás, na prisão, embora com registo na 3.ª Conservatória do Registo Civil de Lisboa, que casou, a 22 de Fevereiro de 1949, com Maria Barroso.
Predicted: Soares married Maria de Jesus Barroso Soares, an actress, on 22 February 1949, while in the Aljube prison, at the 3rd Conservatory of the Civil Register of Lisbon.

Source: ↓ 0.28
Target: ↓0, 28
Predicted: ↓ 0.28

Source: I'm gonna have to take it out kamikaze style.
Target: Vou ter de avançar à kamikaze.
Predicted: I'm gonna have to take it out kamikaze style.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: 0it [00:00, ?it/s]


--- Exemplos de teste ---

Source: We have not forgotten that Cyprus suffered military invasion 30 years ago, at a time when the international situation was admittedly tense.
Target: Pela nossa parte, não esquecemos que Chipre foi invadida militarmente há 30 anos, num contexto internacional evidentemente difícil. Que foi, e continua a estar, dividida em duas.
Predicted: We have not forgotten that Cyprus suffered military invasion 30 years ago, at a time when the international situation was admittedly tense.

Source: No more risking your neck chasing souls.
Target: Vais parar de caçar almas.
Predicted: No more risking your neck chasing souls.

Source: Stop it, both of you.
Target: Parem, os dois.
Predicted: Stop it, both of you.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       avg_test_bleu       │     4.117154598236084     │
└───────────────────────────┴───────────────────────────┘

## Overfit em algumas amostras

Antes de treinar o modelo no dataset todo, faremos overfit do
modelo em poucas de treino para verificar se loss vai para próximo de 0. Isso serve para depurar se a implementação do modelo está correta.

Podemos também medir se a acurácia neste minibatch chega perto de 100%. Isso serve para depurar se nossa função que mede a acurácia está correta.

Nota: se treinarmos por muitas épocas (ex: 500) é possivel que a loss vá para zero mesmo com bugs na implementação. O ideal é que a loss chege próxima a zero antes de 100 épocas.

In [ ]:
trainer = pl.Trainer(gpus=1,
                     precision=16,
                     max_epochs=30,
                     check_val_every_n_epoch=10,
                     enable_checkpointing=False,  # Disable checkpoint saving
                     overfit_batches=0.005)

# Dataset usando apenas um batch de amostras de treino.
dataset_debug = MyDataset(text_pairs=x_train,
                          tokenizer=tokenizer,
                          source_max_length=source_max_length,
                          target_max_length=target_max_length)

debug_dataloader = DataLoader(dataset_debug, batch_size=batch_size,
                              shuffle=False, num_workers=0)

model = T5Finetuner(tokenizer=tokenizer,
                    train_dataloader=debug_dataloader,
                    val_dataloader=debug_dataloader,
                    test_dataloader=None,
                    learning_rate=learning_rate,
                    target_max_length=target_max_length)

trainer.fit(model)
del model  # Para não ter estouro de mémoria da GPU

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit None Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type                       | Params
-----------------------------------------------------
0 | model | T5ForConditionalGeneration | 60.5 M
-----------------------------------------------------
60.5 M    Trainable params
0         Non-trainable params
60.5 M    Total params
121.013   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]


--- Exemplos de validação ---

Source: ==References==* Hanssens, M. 2005.
Target: ==Referências==* Hanssens, M. 2005.
Predicted: ==References==* Hanssens, M. 2005.

Source: Not a day goes by I don't think about you.
Target: Não passa um único dia sem que eu pense em ti.
Predicted: Not a day goes by I don't think about you.

Source: Like, but he was explaining it to us like joyful.
Target: Estava a explicar-nos aquilo de uma forma alegre.
Predicted: Like, but he was explaining it to us like joyful.


...

Validation: 0it [00:00, ?it/s]


--- Exemplos de validação ---

Source: ==References==* Hanssens, M. 2005.
Target: ==Referências==* Hanssens, M. 2005.
Predicted: ==Referências==* Hanssens, M. 2005.

Source: Not a day goes by I don't think about you.
Target: Não passa um único dia sem que eu pense em ti.
Predicted: Não quero saber.

Source: Like, but he was explaining it to us like joyful.
Target: Estava a explicar-nos aquilo de uma forma alegre.
Predicted: Estava a explicar-nos aquilo de uma forma alegre.


Validation: 0it [00:00, ?it/s]


--- Exemplos de validação ---

Source: ==References==* Hanssens, M. 2005.
Target: ==Referências==* Hanssens, M. 2005.
Predicted: ==Referências==* Hanssens, M. 2005.

Source: Not a day goes by I don't think about you.
Target: Não passa um único dia sem que eu pense em ti.
Predicted: Não vemos nada de errado.

Source: Like, but he was explaining it to us like joyful.
Target: Estava a explicar-nos aquilo de uma forma alegre.
Predicted: Estava a explicar-nos aquilo de uma forma alegre.


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


## Treinamento e Validação no dataset todo

In [ ]:
max_epochs = 5

checkpoint_path = 'drive/My Drive/checkpoints/epoch=1.ckpt'
checkpoint_dir = os.path.dirname(os.path.abspath(checkpoint_path))
os.makedirs(checkpoint_dir, exist_ok=True)
print(f'Files in {checkpoint_dir}: {os.listdir(checkpoint_dir)}')
print(f'Saving checkpoints to {checkpoint_dir}')

checkpoint_callback = ModelCheckpoint(dirpath=checkpoint_dir,
                                      save_top_k=-1)  # Keeps all checkpoints.

resume_from_checkpoint = None
if os.path.exists(checkpoint_path):
    print(f'Restoring checkpoint: {checkpoint_path}')
    resume_from_checkpoint = checkpoint_path

trainer = pl.Trainer(gpus=1,
                     precision=16,
                     max_epochs=max_epochs,
                     check_val_every_n_epoch=1,
                     accumulate_grad_batches=accumulate_grad_batches,
                     callbacks=[checkpoint_callback],
                     enable_progress_bar=True)

model = T5Finetuner(tokenizer=tokenizer,
                    train_dataloader=train_dataloader,
                    val_dataloader=val_dataloader,
                    test_dataloader=test_dataloader,
                    learning_rate=learning_rate,
                    target_max_length=target_max_length)

trainer.fit(model, ckpt_path=resume_from_checkpoint)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit None Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Files in /content/drive/My Drive/checkpoints: []
Saving checkpoints to /content/drive/My Drive/checkpoints


Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type                       | Params
-----------------------------------------------------
0 | model | T5ForConditionalGeneration | 60.5 M
-----------------------------------------------------
60.5 M    Trainable params
0         Non-trainable params
60.5 M    Total params
121.013   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]


--- Exemplos de validação ---

Source: Soares married Maria de Jesus Barroso Soares, an actress, on 22 February 1949, while in the Aljube prison, at the 3rd Conservatory of the Civil Register of Lisbon.
Target: Foi, aliás, na prisão, embora com registo na 3.ª Conservatória do Registo Civil de Lisboa, que casou, a 22 de Fevereiro de 1949, com Maria Barroso.
Predicted: Soares married Maria de Jesus Barroso Soares, an actress, on 22 February 1949, while in the Aljube prison, at the 3rd Conservatory of the Civil Register of Lisbon.

Source: ↓ 0.28
Target: ↓0, 28
Predicted: ↓ 0.28

Source: I'm gonna have to take it out kamikaze style.
Target: Vou ter de avançar à kamikaze.
Predicted: I'm gonna have to take it out kamikaze style.


Training: 0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/call.py:48: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


## Após treinado, avaliamos o modelo no dataset de teste

É importante que essa avaliação seja feita poucas vezes para evitar "overfit manual" no dataset de teste.

In [ ]:
trainer.test(model)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: 0it [00:00, ?it/s]


--- Exemplos de teste ---

Source: We have not forgotten that Cyprus suffered military invasion 30 years ago, at a time when the international situation was admittedly tense.
Target: Pela nossa parte, não esquecemos que Chipre foi invadida militarmente há 30 anos, num contexto internacional evidentemente difícil. Que foi, e continua a estar, dividida em duas.
Predicted: Não podemos negar que Cyprus suprisse o regime militar de 30 anos ago, a menos que a situação interna não seja admissível.

Source: No more risking your neck chasing souls.
Target: Vais parar de caçar almas.
Predicted: Não vás apanhar os meus amigos.

Source: Stop it, both of you.
Target: Parem, os dois.
Predicted: Vê-te, não é?


In [ ]:
trainer.test(model)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: 0it [00:00, ?it/s]

## Comentários

---
Comentários:
Ferramenta: Gemini (Google DeepMind)
Utilização: Utilizado para desenvolvimento de código, implementação dos métodos da classe Dataset (MyDataset) e do módulo PyTorch Lightning (T5Finetuner), além de suporte na integração e cálculo da métrica SacreBLEU.
---